## Data Collection & Dataset Splitting

### Objectives
* Setup Kaggle authentication and download the official Cherry Leaves dataset from Kaggle.
* Inspect and clean dataset by removing non-image files.
* Split dataset randomly into **Train (70%)**, **Validation (10%)**, and **Test (20%)** sets.
* Plot and save the image distribution across splits to verify class balance.

### Inputs
* `kaggle.json` API authentication file located in project root.
* Kaggle Dataset: `codeinstitute/cherry-leaves`

### Outputs
* `inputs/cherry_leaves/` directory containing `train`, `validation`, and `test` folders split into `healthy` and `powdery_mildew`.
* `outputs/v1/class_distribution.png` visual plot.

In [14]:
import os
import glob
import random
import shutil
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Define project directory paths
PROJECT_DIR = os.getcwd()
INPUTS_DIR = os.path.join(PROJECT_DIR, 'inputs')
RAW_DATA_DIR = os.path.join(INPUTS_DIR, 'raw_data')
SPLIT_DATA_DIR = os.path.join(INPUTS_DIR, 'cherry_leaves')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')

# Create necessary directories
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(SPLIT_DATA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("Directories initialized successfully.")

Directories initialized successfully.


In [12]:
import os

# If current directory is 'jupyter_notebooks', navigate one level up to project root
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'jupyter_notebooks':
    PROJECT_DIR = os.path.abspath(os.path.join(current_dir, '..'))
else:
    PROJECT_DIR = current_dir

INPUTS_DIR = os.path.join(PROJECT_DIR, 'inputs')
RAW_DATA_DIR = os.path.join(INPUTS_DIR, 'raw_data')
SPLIT_DATA_DIR = os.path.join(INPUTS_DIR, 'cherry_leaves')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')

print(f"Project Root Directory: {PROJECT_DIR}")

Project Root Directory: d:\books\code\code-institut\PROJECTS\cherry tree leaves


In [9]:
%pip install kaggle


   ------ ---------------------------------  2/13 [tqdm]
   ------ ---------------------------------  2/13 [tqdm]
   ------ ---------------------------------  2/13 [tqdm]
   --------- ------------------------------  3/13 [pyyaml]
   ------------ ---------------------------  4/13 [python-slugify]
   --------------- ------------------------  5/13 [python-dotenv]
   ------------------ ---------------------  6/13 [fastjsonschema]
   --------------------- ------------------  7/13 [mdit-py-plugins]
   --------------------- ------------------  7/13 [mdit-py-plugins]
   --------------------- ------------------  7/13 [mdit-py-plugins]
   --------------------- ------------------  7/13 [mdit-py-plugins]
   --------------------- ------------------  7/13 [mdit-py-plugins]
   --------------------- ------------------  7/13 [mdit-py-plugins]
   ------------------------ ---------------  8/13 [kagglesdk]
   ------------------------ ---------------  8/13 [kagglesdk]
   ------------------------ ---------

In [9]:
# Download Kaggle Dataset
import kaggle

dataset_name = "codeinstitute/cherry-leaves"
print(f"Downloading dataset '{dataset_name}'...")

kaggle.api.dataset_download_files(dataset_name, path=RAW_DATA_DIR, unzip=True)
print("Download and extraction complete.")

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
Dataset URL: https://www.kaggle.com/datasets/codeinstitute/cherry-leaves
Download and extraction complete.


In [8]:
# Verify and remove non-image files
valid_extensions = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')
removed_count = 0

for root, _, files in os.walk(RAW_DATA_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        if not file.endswith(valid_extensions):
            os.remove(file_path)
            removed_count += 1
            print(f"Removed non-image file: {file_path}")

print(f"Data cleaning complete. Total invalid files removed: {removed_count}")

Data cleaning complete. Total invalid files removed: 0


In [15]:
def split_dataset(raw_dir, target_dir, train_ratio=0.70, val_ratio=0.10, test_ratio=0.20, seed=42):
    """
    Splits images from raw directory into train, validation, and test folders.
    """
    assert np.isclose(train_ratio + val_ratio + test_ratio, 1.0), "Ratios must sum to 1.0"
    random.seed(seed)
    
    # Locate category folders (healthy, powdery_mildew)
    categories = [d for d in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir, d))]
    
    for category in categories:
        category_raw_path = os.path.join(raw_dir, category)
        images = [f for f in os.listdir(category_raw_path) if f.endswith(valid_extensions)]
        random.shuffle(images)
        
        n_total = len(images)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        
        splits = {
            'train': images[:n_train],
            'validation': images[n_train:n_train + n_val],
            'test': images[n_train + n_val:]
        }
        
        for split_name, split_files in splits.items():
            split_folder = os.path.join(target_dir, split_name, category)
            os.makedirs(split_folder, exist_ok=True)
            
            for file_name in split_files:
                src_path = os.path.join(category_raw_path, file_name)
                dst_path = os.path.join(split_folder, file_name)
                shutil.copyfile(src_path, dst_path)
                
        print(f"Category '{category}': {n_total} total -> Train: {len(splits['train'])}, Val: {len(splits['validation'])}, Test: {len(splits['test'])}")

# Run splitting function
split_dataset(
    raw_dir=os.path.join(RAW_DATA_DIR, 'cherry-leaves'), # Adjust subfolder name if extracted directly
    target_dir=SPLIT_DATA_DIR
)

Category 'healthy': 2104 total -> Train: 1472, Val: 210, Test: 422
Category 'powdery_mildew': 2104 total -> Train: 1472, Val: 210, Test: 422


In [16]:
# Calculate dataset distribution across splits and classes
records = []

for split in ['train', 'validation', 'test']:
    for label in ['healthy', 'powdery_mildew']:
        folder_path = os.path.join(SPLIT_DATA_DIR, split, label)
        count = len(os.listdir(folder_path)) if os.path.exists(folder_path) else 0
        records.append({'Split': split.capitalize(), 'Label': label.replace('_', ' ').title(), 'Count': count})

df_dist = pd.DataFrame(records)
print(df_dist)

# Plot class distribution
plt.figure(figsize=(9, 5))
sns.barplot(data=df_dist, x='Split', y='Count', hue='Label', palette='viridis')
plt.title('Cherry Leaves Dataset Distribution across Splits', fontsize=14, fontweight='bold')
plt.xlabel('Dataset Split', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Save plot to outputs/v1
plot_path = os.path.join(OUTPUTS_DIR, 'class_distribution.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Class distribution chart saved to {plot_path}")

        Split           Label  Count
0       Train         Healthy   1472
1       Train  Powdery Mildew   1472
2  Validation         Healthy    210
3  Validation  Powdery Mildew    210
4        Test         Healthy    422
5        Test  Powdery Mildew    422


<Figure size 900x500 with 1 Axes>

Class distribution chart saved to d:\books\code\code-institut\PROJECTS\cherry tree leaves\jupyter_notebooks\outputs\v1\class_distribution.png
